# 02 · Fields, Types & Constraints

**Module 15 – Pydantic: Data Validation & Settings** · Estimated time: 55 min

### 🎯 What you will learn
- `Field()`: defaults, `default_factory`, numeric/string/collection **constraints**, descriptions & examples
- The ✅ modern `Annotated[...]` style for reusable constrained types
- Built-in types: numbers, `Decimal`, strings, `bytes`, dates & times, `UUID`, `Enum`, `Literal`, paths, IPs
- Pydantic's special types: `EmailStr`, `HttpUrl`, `SecretStr`, `PositiveInt`, `NonNegativeFloat`, `FutureDate`, `Json`, `ByteSize`...
- Collections: `list`, `set`, `tuple`, `dict`, nested containers
- ⚠️ `conint()`/`constr()` (old style) vs `Annotated` (new style)

In [1]:
from datetime import date, datetime, timedelta
from decimal import Decimal
from enum import Enum
from typing import Annotated, Literal
from uuid import UUID, uuid4
from pydantic import BaseModel, Field, ValidationError

def show_errors(model, **data):
    try:
        print("✅", model(**data))
    except ValidationError as e:
        for err in e.errors():
            print("❌", ".".join(map(str, err["loc"])), "→", err["msg"])

## 1. `Field()`

In [2]:
class Product(BaseModel):
    sku: str = Field(min_length=3, max_length=12, pattern=r"^[A-Z0-9-]+$", description="Stock keeping unit")
    name: str = Field(min_length=1, max_length=100, examples=["Fountain pen"])
    price: Decimal = Field(gt=0, max_digits=10, decimal_places=2)
    discount: float = Field(default=0.0, ge=0, le=0.9)
    quantity: int = Field(default=1, ge=1, multiple_of=1)
    tags: list[str] = Field(default_factory=list, max_length=5)        # factory: a new list every time
    created: datetime = Field(default_factory=datetime.now)            # computed at creation

show_errors(Product, sku="PEN-01", name="Pen", price="12.50")
show_errors(Product, sku="pen 01", name="", price="-1", discount=2, tags=list("abcdef"))

✅ sku='PEN-01' name='Pen' price=Decimal('12.50') discount=0.0 quantity=1 tags=[] created=datetime.datetime(2026, 10, 20, 19, 46, 4, 128609)
❌ sku → String should match pattern '^[A-Z0-9-]+$'
❌ name → String should have at least 1 character
❌ price → Input should be greater than 0
❌ discount → Input should be less than or equal to 0.9
❌ tags → List should have at most 5 items after validation, not 6


| Constraint | Applies to | Meaning |
|---|---|---|
| `gt`, `ge`, `lt`, `le` | numbers, dates | >, ≥, <, ≤ |
| `multiple_of` | numbers | divisible by |
| `max_digits`, `decimal_places` | `Decimal` | precision |
| `min_length`, `max_length` | `str`, `list`, `set`, `dict`... | size |
| `pattern` | `str` | regular expression |
| `strict` | any | no coercion for this field |
| `allow_inf_nan` | floats | allow `inf`/`nan` |
| `frozen` | any | can't be reassigned |
| `exclude` | any | left out of `model_dump` |

Metadata for docs/JSON Schema: `title`, `description`, `examples`, `deprecated`, `json_schema_extra`.

### Required fields with `Field`
`Field(...)` (Ellipsis) or simply no default means **required**: `name: str = Field(min_length=1)`.

## 2. ✅ `Annotated`: reusable constrained types

Instead of repeating `Field(...)` everywhere, define a **type** once:

In [3]:
Username = Annotated[str, Field(min_length=3, max_length=20, pattern=r"^[a-z0-9_]+$")]
Percentage = Annotated[float, Field(ge=0, le=100)]
PositiveMoney = Annotated[Decimal, Field(gt=0, decimal_places=2)]

class Account(BaseModel):
    owner: Username
    backup_owner: Username | None = None
    interest: Percentage
    balance: PositiveMoney

show_errors(Account, owner="asha_31", interest=4.5, balance="1000.00")
show_errors(Account, owner="A!", backup_owner="x", interest=150, balance="10.555")

✅ owner='asha_31' backup_owner=None interest=4.5 balance=Decimal('1000.00')
❌ owner → String should have at least 3 characters
❌ backup_owner → String should have at least 3 characters
❌ interest → Input should be less than or equal to 100
❌ balance → Decimal input should have no more than 2 decimal places


`StringConstraints` adds string transformations:

In [4]:
from pydantic import StringConstraints
CountryCode = Annotated[str, StringConstraints(strip_whitespace=True, to_upper=True, min_length=2, max_length=2)]

class Address(BaseModel):
    city: str
    country: CountryCode

print(Address(city="Pune", country="  in "))

city='Pune' country='IN'


> ⚠️ **Old style (still works, discouraged):** `conint(gt=0)`, `constr(min_length=3)`, `confloat`, `conlist`. Prefer `Annotated[int, Field(gt=0)]`, which type checkers understand.

## 3. Built-in Python types Pydantic understands

In [5]:
class Color(str, Enum):
    RED = "red"
    GREEN = "green"

class Everything(BaseModel):
    i: int
    f: float
    d: Decimal
    b: bool
    s: str
    raw: bytes
    day: date
    when: datetime
    span: timedelta
    uid: UUID
    color: Color
    size: Literal["S", "M", "L"]
    pair: tuple[int, str]
    unique: set[int]
    scores: dict[str, float]
    matrix: list[list[int]]

e = Everything(
    i="1", f="2.5", d="0.1", b="on", s="text", raw="bytes!", day="2026-10-20",
    when="2026-10-20T14:30:00+05:30", span="PT1H30M", uid=str(uuid4()), color="green",
    size="M", pair=["7", "seven"], unique=[1, 1, 2], scores={"math": "91.5"}, matrix=[[1, "2"], [3]],
)
for name, value in e:
    print(f"{name:<7} {type(value).__name__:<10} {value!r}")

i       int        1
f       float      2.5
d       Decimal    Decimal('0.1')
b       bool       True
s       str        'text'
raw     bytes      b'bytes!'
day     date       datetime.date(2026, 10, 20)
when    datetime   datetime.datetime(2026, 10, 20, 14, 30, tzinfo=TzInfo(19800))
span    timedelta  datetime.timedelta(seconds=5400)
uid     UUID       UUID('dbfbc335-1696-4f07-a03b-06e546b7b851')
color   Color      <Color.GREEN: 'green'>
size    str        'M'
pair    tuple      (7, 'seven')
unique  set        {1, 2}
scores  dict       {'math': 91.5}
matrix  list       [[1, 2], [3]]


Note how JSON-ish inputs (`"PT1H30M"` ISO durations, ISO datetimes with offsets, lists for tuples/sets) are converted to proper Python types.

## 4. Pydantic's special types

In [6]:
from pydantic import (EmailStr, HttpUrl, AnyUrl, SecretStr, PositiveInt, NonNegativeFloat,
                      FutureDate, PastDate, AwareDatetime, ByteSize, IPvAnyAddress, Json, conint)

class Signup(BaseModel):
    email: EmailStr                      # needs: pip install "pydantic[email]"
    website: HttpUrl | None = None
    password: SecretStr                  # hidden in repr/logs
    age: PositiveInt
    rating: NonNegativeFloat = 0
    birthday: PastDate
    trial_ends: FutureDate
    last_login: AwareDatetime | None = None   # must include a timezone
    upload_limit: ByteSize = "10MB"
    ip: IPvAnyAddress | None = None
    preferences: Json[dict[str, bool]] = "{}"  # a JSON STRING that's parsed and validated

s = Signup(email="Asha@Example.COM", website="https://asha.dev/blog?x=1", password="hunter2",
           age=31, birthday="1995-07-19", trial_ends="2030-01-01", upload_limit="1.5GiB",
           ip="192.168.1.10", preferences='{"newsletter": true}')
print(s)
print(s.email, "|", s.website.host, s.website.path, "|", s.password, s.password.get_secret_value())
print(s.upload_limit, "bytes =", s.upload_limit.human_readable(), "|", s.preferences)

email='Asha@example.com' website=HttpUrl('https://asha.dev/blog?x=1') password=SecretStr('**********') age=31 rating=0 birthday=datetime.date(1995, 7, 19) trial_ends=datetime.date(2030, 1, 1) last_login=None upload_limit=1610612736 ip=IPv4Address('192.168.1.10') preferences={'newsletter': True}
Asha@example.com | asha.dev /blog | ********** hunter2
1610612736 bytes = 1.5GiB | {'newsletter': True}


In [7]:
show_errors(Signup, email="not-an-email", website="ftp://x", password="x", age=0,
            birthday="2999-01-01", trial_ends="2000-01-01", last_login="2026-10-20T10:00:00")

❌ email → value is not a valid email address: An email address must have an @-sign.
❌ website → URL scheme should be 'http' or 'https'
❌ age → Input should be greater than 0
❌ birthday → Date should be in the past
❌ trial_ends → Date should be in the future
❌ last_login → Input should have timezone info


More types live in **`pydantic-extra-types`** (phone numbers, country codes, currencies, colours, payment cards, coordinates...).

## 5. Collections & nested containers

In [8]:
class Gradebook(BaseModel):
    students: list[str] = Field(min_length=1)
    grades: dict[str, list[Annotated[int, Field(ge=0, le=100)]]]
    weights: tuple[float, float, float]
    flags: frozenset[str] = frozenset()

show_errors(Gradebook, students=["asha"], grades={"asha": [90, "85"]}, weights=(0.3, 0.3, 0.4))
show_errors(Gradebook, students=[], grades={"asha": [90, 101]}, weights=(1, 2))

✅ students=['asha'] grades={'asha': [90, 85]} weights=(0.3, 0.3, 0.4) flags=frozenset()
❌ students → List should have at least 1 item after validation, not 0
❌ grades.asha.1 → Input should be less than or equal to 100
❌ weights.2 → Field required


Constraints inside `list[Annotated[int, Field(...)]]` apply to **each item**.

## 6. Field aliases and names (preview)

In [9]:
class Repo(BaseModel):
    full_name: str = Field(alias="fullName")     # the key used in input data
    stars: int = Field(alias="stargazers_count")

r = Repo.model_validate({"fullName": "python/cpython", "stargazers_count": 70000})
print(r.full_name, r.stars, r.model_dump(by_alias=True))

python/cpython 70000 {'fullName': 'python/cpython', 'stargazers_count': 70000}


Aliases, alias generators (camelCase) and serialization names are covered in lesson 04.

## 📝 Summary
- `Field()` adds defaults, `default_factory`, constraints and documentation.
- ✅ Build reusable constrained types with `Annotated[T, Field(...)]` / `StringConstraints`.
- Pydantic understands standard types (dates, `Decimal`, `UUID`, `Enum`, `Literal`, collections) and adds special ones (`EmailStr`, `HttpUrl`, `SecretStr`, `PositiveInt`, `FutureDate`, `Json[...]`...).

## 🏋️ Practice
1. Define `Phone = Annotated[str, StringConstraints(pattern=r"^\+?[0-9]{10,13}$")]` and a `Contact` model using it.
2. Create a `Booking` model: `guests: int` (1–10), `check_in: FutureDate`, `nights: int` (≥1), `room: Literal["single", "double", "suite"]`, `email: EmailStr`.
3. Model a GitHub API repo response with aliases for `stargazers_count`, `forks_count`, `open_issues_count`.